<a href="https://colab.research.google.com/github/Nischay-verma/Capstone_Project-6_CHAT_BOT_LLM_Deep_Learning_for_NLP/blob/main/Capstone_Project_3_AI_Engineering_CHAT_BOT_LLM_Deep_Learning_for_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Capstone Project: Industry-Specific LLM Bot

##### **Contribution** — Individual
##### **Name**          - Nischay Verma

# **GitHub Link -**

https://github.com/Nischay-verma/Capstone_Project-6_CHAT_BOT_LLM_Deep_Learning_for_NLP

##**Industry-Specific LLM Bot – Capstone Project**

This capstone project focuses on developing an Industry-Specific Large Language Model (LLM) Bot using pre-trained models from platforms like Hugging Face. The chatbot will be designed to answer user queries and provide insights related to a selected industry such as healthcare, finance, education, or e-commerce.

Students will work on data collection, model selection, customization, chatbot development, and deployment using technologies like Python, NLP, Transformers, and Streamlit/Flask. The project aims to enhance practical skills in Artificial Intelligence, Natural Language Processing, and Large Language Models while providing industry-specific knowledge and real-world problem-solving experience.

## Project Summary

This project, **Industry-Specific LLM Bot – Capstone Project**, focuses on developing a Large Language Model (LLM) based chatbot tailored for a specific industry (e.g., retail banking, as indicated by the dataset used). The core idea is to leverage pre-trained models, such as those available on Hugging Face, to create a conversational AI that can answer user queries and provide insights within that chosen industry.

The development process involves several key stages:

*   **Data Collection**: Gathering relevant industry-specific data for training.
*   **Model Selection**: Choosing appropriate pre-trained LLM models (e.g., `google/flan-t5-small`).
*   **Customization**: Fine-tuning the selected model on the collected dataset.
*   **Chatbot Development**: Implementing the chatbot logic to process user input and generate responses.
*   **Deployment**: Utilizing frameworks like Gradio to create an interactive user interface for the chatbot.

Technologies employed include Python, Natural Language Processing (NLP) techniques, Transformer models, and libraries like `transformers`, `datasets`, `scikit-learn`, and `gradio`. The project aims to provide practical experience in AI, NLP, and LLMs while offering a solution for real-world industry-specific problem-solving.

In [1]:
pip install transformers datasets scikit-learn gradio


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd

# Load the CSV file
df = pd.read_csv("/content/drive/MyDrive/AlmaBetter masters assesment/AI Engineering/Module 3 AI Engineering/LLM/bitext-retail-banking-llm-chatbot-training-dataset.csv")

# Keep only needed columns and drop missing values
df_clean = df[['instruction', 'response']].dropna()
df_clean = df_clean[df_clean['instruction'].str.strip() != ""]
df_clean = df_clean[df_clean['response'].str.strip() != ""]
df_clean = df_clean.drop_duplicates(subset=['instruction', 'response'])

# Rename for model input/output
df_clean = df_clean.rename(columns={"instruction": "input", "response": "output"})

# Preview
df_clean.sample(3)


,input,output
24333,I got to retrieve a Master Card,I regret that the inconvenience you're facing ...
7032,i cannot pay for my mortgage can ucancel it,I'm sorry to hear that you're facing difficult...
17514,"I need to talk to fucking customer support, wi...",I regret that any frustration or inconvenience...


In [4]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df_clean, test_size=0.1, random_state=42)


In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [6]:
from datasets import Dataset

# Convert to Hugging Face format
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Tokenization function
def tokenize(example):
    input_enc = tokenizer(example["input"], padding="max_length", truncation=True, max_length=128)
    target_enc = tokenizer(example["output"], padding="max_length", truncation=True, max_length=128)
    input_enc["labels"] = target_enc["input_ids"]
    return input_enc

# Tokenize datasets
train_tokenized = train_dataset.map(tokenize)
test_tokenized = test_dataset.map(tokenize)


Map:   0%|          | 0/22990 [00:00<?, ? examples/s]

Map:   0%|          | 0/2555 [00:00<?, ? examples/s]

In [7]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

training_args = Seq2SeqTrainingArguments(
    output_dir="./banking_bot",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    predict_with_generate=True,
    num_train_epochs=3,
    logging_dir='./logs',
    eval_strategy="epoch"
)

collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    data_collator=collator
)

trainer.train()

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss
1,1.044619,0.853859
2,0.921312,0.779668
3,0.894955,0.761588


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=17244, training_loss=1.0451160821138195, metrics={'train_runtime': 2559.565, 'train_samples_per_second': 26.946, 'train_steps_per_second': 6.737, 'total_flos': 3205217027358720.0, 'train_loss': 1.0451160821138195, 'epoch': 3.0})

In [8]:
# prompt: give testing accuracy training accuracy and validation accuracy

# After trainer.train()
import json

# Function to extract metrics from logs
def get_metrics(log_history):
    train_loss = None
    eval_loss = None
    # Find the final training loss
    for log in log_history:
        if 'loss' in log:
            train_loss = log['loss']
        if 'eval_loss' in log:
            eval_loss = log['eval_loss']
    return train_loss, eval_loss

train_loss, eval_loss = get_metrics(trainer.state.log_history)

print(f"Training Loss (last logged): {train_loss}")
print(f"Validation Loss (last epoch): {eval_loss}")

# To get evaluation metrics on the test set after training
eval_results = trainer.evaluate(test_tokenized)
print(f"Test Evaluation Results: {eval_results}")

# Note: For accuracy in seq2seq models, it's usually not a single number
# like classification accuracy. Metrics like ROUGE, BLEU are common.
# If you need specific accuracy-like metrics, you would need to implement
# or use a library for sequence generation evaluation metrics and potentially
# override the compute_metrics function in the Trainer or perform manual evaluation.
# The default evaluation in Seq2SeqTrainer provides loss.



Training Loss (last logged): 0.8949552001953125
Validation Loss (last epoch): 0.7615882754325867


Training Loss,Validation Loss,Epoch
0.894955,0.761588,3


Test Evaluation Results: {'eval_loss': 0.7615882754325867}


In [9]:
model.save_pretrained("banking-chatbot")
tokenizer.save_pretrained("banking-chatbot")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('banking-chatbot/tokenizer_config.json', 'banking-chatbot/tokenizer.json')

In [10]:
import torch

def chat(prompt):
    # Set device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Tokenize and move tensors to the same device as model
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Generate response
    with torch.no_grad():
        output = model.generate(**inputs, max_length=100)

    return tokenizer.decode(output[0], skip_special_tokens=True)


In [11]:
chat("How can I activate my debit card?")


'I\'m here to assist you with activating your debit card. Activating your card is a simple process. Here\'s what you need to do: 1. Visit our website at Company Website URL. 2. Look for the "Activate Card" or "Card Activation" option. 3. Click on it to access the activation instructions. 4. You will be prompted to enter your card details, such as the card number, expiration date'

In [12]:
import gradio as gr

In [13]:
interface = gr.Interface(
    fn=chat,          # The function to wrap with a UI
    inputs=gr.Textbox(label="Your banking question"), # Input component: a textbox for the user's prompt
    outputs=gr.Textbox(label="Chatbot response"), # Output component: a textbox for the chatbot's response
    title="Retail Banking Chatbot" # Title for the Gradio interface
)

In [14]:
import gradio as gr

interface = gr.Interface(
    fn=chat,
    inputs=gr.Textbox(label="Your banking question"),
    outputs=gr.Textbox(label="Chatbot response"),
    title="Retail Banking Chatbot"
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9d2621ceb90e1db638.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
